# LSTM Sanity Checks (CLASSIFIER)

Runs six checks on the *same* held-out test set against an already-trained GELSTM checkpoint, and tabulates them side-by-side:

1. **Baseline** — original config (`use_time_delta=True`, full visit history, true temporal order).
2. **Shuffled visit order** — visits per subject randomly permuted at eval. Δt rides with its original visit so the Δt distribution is preserved while order is destroyed.
3. **Δt removed** — `use_time_delta=False` at eval.
4. **Fixed sequence length (first-2 visits)** — uses `LongitudinalSubjectDataset(max_visits=2, require_full_window=True)`.
5. **Metadata-only baseline** — copied across from `SANITY_TIME_METADATA_BASELINE.ipynb` for cross-reference.
6. **Visit-count confound** — cohort-composition and fixed-cohort AUC-vs-N via `common/visit_confound.py`, to separate real added-visit information from class-balance/cohort-size artifacts in the raw AUC-vs-N table.

*All LSTM rows use the same trained checkpoint to isolate the effect of each ablation.*

In [1]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


In [2]:
# Parameters
EXPERIMENT_ID = "sanity-gelstm-ablations"
MODE = "sanity"
MODEL = "GELSTM"
DATASET = "DELCODE_WHOLE_BRAIN"
SEED = 42
GAAE_CHECKPOINT_PATH = None
THRESHOLD_MODE = None
FIXED_THRESHOLD = None
WANDB_ENABLED = True
OUTPUT_DIR = "outputs/sanity-gelstm-ablations"
RESOLVED_CONFIG = {"epochs": 100, "lr": 0.001, "weight_decay": 0.0, "rnn_type": "lstm", "batch_size": 16, "grad_clip": 1.0, "early_stopping_patience": 20, "use_scheduler": True, "seed": 42, "threshold_mode": "youden", "fixed_threshold": 0.5, "lr_factor": 0.5, "lr_patience": 5, "lr_min": 1e-06, "classifier_norm": "none", "use_time_delta": True, "zero_time_delta": False, "graph_pool": "mean", "dim_filter": None, "shuffle_order": False, "shuffle_rng": None}
RUN_DIR = "/mnt/e/fyassine/ad-early-detection/CLASSIFIER/outputs/sanity-gelstm-ablations/runs/dainty-dream-6-nogit-2026-07-03_16-37-52"
RUN_NAME = "dainty-dream-6-nogit-2026-07-03_16-37-52"


In [3]:

import sys
import importlib
from pathlib import Path
import json, os, copy
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

V2_ROOT = REPO_ROOT / 'CLASSIFIER'
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

# Force reload so edits to model source are picked up without restarting the kernel.
import model.GELSTM.utils
import model.GELSTM.train
importlib.reload(model.GELSTM.utils)
importlib.reload(model.GELSTM.train)

from model.GELSTM.models  import GELSTMClassifier
from model.GELSTM.dataset import LongitudinalSubjectDataset
from model.GELSTM.train   import evaluate, make_batches
from SHARED.seeding       import set_seed
from SHARED.sanity        import run_full_audit

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [4]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.DELCODE.src.splitting.load_splits import splits_dir, split_csv_paths


import json
from pathlib import Path

from SHARED.sanity import run_full_audit

DATA_VERSION = '__fc_wholebrain_sch200_flat__'
DATA_ROOT    = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
WB_DATA_ROOT = str(DATA_ROOT / 'matrices')
METADATA_DIR = DATA_ROOT / 'metadata'
COHORTS_CSV  = str(METADATA_DIR / 'cohorts_with_scans_on_disk.csv')

SPLIT_CSVS = {
    'train': str(splits_dir('downstream') / 'train.csv'),
    'val':   str(splits_dir('downstream') / 'val.csv'),
    'test':  str(splits_dir('downstream') / 'test.csv'),
}
_ = run_full_audit(SPLIT_CSVS)

GAAE_CONFIG_PATH = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/configs/gaae_delcode_whole_brain.json')

# ── Checkpoint selector ──────────────────────────────────────────────────────
CKPT_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints')
CHECKPOINT_SEARCH_DIRS = sorted(CKPT_ROOT.glob('checkpoints_gelstm*'))

checkpoint_candidates = sorted(
    [(run_dir.name, str(run_dir / f'model_{run_dir.name}.pth'), str(run_dir))
     for ckpt_dir in CHECKPOINT_SEARCH_DIRS
     for run_dir in sorted(Path(ckpt_dir).iterdir()) if run_dir.is_dir()
     if (run_dir / f'model_{run_dir.name}.pth').exists()],
    key=lambda x: x[0],
)
if not checkpoint_candidates:
    raise FileNotFoundError(f'No GELSTM checkpoints found under {CKPT_ROOT}')

print('Available GELSTM checkpoints:')
for i, (name, _, _) in enumerate(checkpoint_candidates):
    print(f'  {i}: {name}')

if GAAE_CHECKPOINT_PATH is not None:
    _t = str(Path(GAAE_CHECKPOINT_PATH).resolve())
    _m = [c for c in checkpoint_candidates if str(Path(c[1]).resolve()) == _t]
    if not _m:
        raise FileNotFoundError(f'GAAE_CHECKPOINT_PATH={GAAE_CHECKPOINT_PATH!r} not among candidates')
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = _m[0]
elif RUN_DIR is not None:
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = checkpoint_candidates[-1]
    print(f'Headless: using latest GELSTM checkpoint {GELSTM_RUN_NAME}')
else:
    selected_idx = int(input('Select checkpoint index: '))
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = checkpoint_candidates[selected_idx]
print(f'Selected: {GELSTM_RUN_NAME}')
print(f'Path:     {GELSTM_CKPT_PATH}')


[SANITY] Split sizes: {'train': 99, 'val': 34, 'test': 34}
[SANITY] Pairwise-disjoint: OK
Available GELSTM checkpoints:
  0: gelstm_2026-05-20_09-54-16
  1: gelstm_fdr_15_2026-05-08_07-29-50
  2: gelstm_fdr_15_2026-05-08_09-53-45
  3: gelstm_wholebrain_2026-06-10_20-37-58
  4: gelstm_wholebrain_2026-06-12_15-44-49
Headless: using latest GELSTM checkpoint gelstm_wholebrain_2026-06-12_15-44-49
Selected: gelstm_wholebrain_2026-06-12_15-44-49
Path:     /mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints/checkpoints_gelstm_whole_brain/gelstm_wholebrain_2026-06-12_15-44-49/model_gelstm_wholebrain_2026-06-12_15-44-49.pth


## Load test set + checkpoint

We build two test datasets: one with **all visits** (used by checks 1-3), one with **first-2-visits and require_full_window=True** (used by check 4).

In [5]:
test_df = pd.read_csv(SPLIT_CSVS['test'])
test_mci = test_df.copy()
print(f'Test subjects (MCI/converter): {len(test_mci)}')

with open(GAAE_CONFIG_PATH) as f:
    hp = json.load(f)
ADJ_K        = hp.get('adjacency_k', 8)
FILE_VARIANT = hp.get('file_variant', 'z_transformed')

test_ds_all = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
)
test_ds_first2 = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
    max_visits=2, require_full_window=True,
)
test_items_all    = [test_ds_all[i]    for i in range(len(test_ds_all))]
test_items_first2 = [test_ds_first2[i] for i in range(len(test_ds_first2))]

Test subjects (MCI/converter): 34


LongitudinalSubjectDataset[v2]: 34 subjects (14 converter, 20 stable MCI)
  Scans per subject: min=1  max=5  mean=2.4


LongitudinalSubjectDataset[v2]: 25 subjects (11 converter, 14 stable MCI)
  Window: first 2 visit(s); require_full_window=True; dropped (insufficient visits)=9
  Scans per subject: min=2  max=2  mean=2.0


In [6]:

def load_gelstm():
    m = GELSTMClassifier(
        in_features=200, gaae_hidden=hp.get('hidden_dim', 128),
        gaae_latent=hp.get('latent_dim', 64),
        gaae_heads=hp.get('num_heads', 2),
        gaae_cond_dim=hp.get('cond_dim', 2),
        gaae_dropout=hp.get('dropout', 0.3),
        lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
        use_time_delta=True, classifier_hidden=64,
    ).to(device)
    state = torch.load(GELSTM_CKPT_PATH, map_location=device)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state, strict=False)
    m.eval()
    print(f'Loaded: {GELSTM_CKPT_PATH}')
    return m


In [7]:

# ► Select a checkpoint from the dropdown in cell 2, then run this cell.
model = load_gelstm()


Loaded: /mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints/checkpoints_gelstm_whole_brain/gelstm_wholebrain_2026-06-12_15-44-49/model_gelstm_wholebrain_2026-06-12_15-44-49.pth


In [8]:

BATCH_SIZE = 16
rng = np.random.default_rng(RANDOM_STATE)
rows = []

def _run(items, **eval_kwargs):
    batches = make_batches(items, BATCH_SIZE, shuffle=False)
    return evaluate(model, batches, device, **eval_kwargs)

# 1. Baseline.
r = _run(test_items_all, use_time_delta=True, shuffle_order=False)
rows.append({'check': '1_baseline', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

# 2. Shuffled visit order (repeat 5x to estimate variance).
aucs = []
for k in range(5):
    r = _run(test_items_all, use_time_delta=True, shuffle_order=True,
             shuffle_rng=np.random.default_rng(RANDOM_STATE + k))
    aucs.append(r['auc'])
rows.append({'check': '2_shuffled_order', 'auc': float(np.mean(aucs)), 'sens': float('nan'),
             'spec': float('nan'), 'auc_std': float(np.std(aucs))})

# 3. Δt zeroed-out (keeps input shape intact; only the signal is removed).
r = _run(test_items_all, use_time_delta=True, zero_time_delta=True, shuffle_order=False)
rows.append({'check': '3_no_delta_t', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

# 4. Fixed sequence length (first 2 visits, full-window).
r = _run(test_items_first2, use_time_delta=True, shuffle_order=False)
rows.append({'check': '4_first_2_visits', 'auc': r['auc'], 'sens': r['sensitivity'],
             'spec': r['specificity'], 'n_subjects': len(test_items_first2)})

pd.DataFrame(rows)


,check,auc,sens,spec,auc_std,n_subjects
0,1_baseline,0.907143,0.857143,0.850000,NaN,NaN
1,2_shuffled_order,0.848571,NaN,NaN,0.025813,NaN
2,3_no_delta_t,0.907143,0.857143,0.850000,NaN,NaN
3,4_first_2_visits,0.876623,1.000000,0.642857,NaN,25.0


## Cross-reference: metadata-only baseline

Reuse the result table from `SANITY_TIME_METADATA_BASELINE.ipynb` to anchor row 5 of the sanity table — set the value manually after running that notebook.

In [9]:

METADATA_BASELINE_JSON = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/metadata_baseline_results.json')

if METADATA_BASELINE_JSON.exists():
    with open(METADATA_BASELINE_JSON) as f:
        _meta = json.load(f)
    META_ONLY_CV_AUC   = _meta['cv_auc_mean']
    META_ONLY_TEST_AUC = _meta['test_auc']
    print(f'Loaded metadata baseline from {METADATA_BASELINE_JSON}')
    print(f"  best model : {_meta['best_model']}")
    print(f"  cv_auc     : {META_ONLY_CV_AUC:.4f} ± {_meta['cv_auc_std']:.4f}")
    print(f"  test_auc   : {META_ONLY_TEST_AUC:.4f}")
else:
    META_ONLY_CV_AUC   = float('nan')
    META_ONLY_TEST_AUC = float('nan')
    print(f'WARNING: {METADATA_BASELINE_JSON} not found.')
    print('Run SANITY_TIME_METADATA_BASELINE.ipynb first to generate it.')

rows.append({
    'check':   '5_metadata_only_baseline',
    'auc':     META_ONLY_TEST_AUC,
    'cv_auc':  META_ONLY_CV_AUC,
})
summary = pd.DataFrame(rows)

# Display with — in place of NaN so the table is easier to read.
summary.fillna('—').style.set_caption('GELSTM Sanity Checks')


Run SANITY_TIME_METADATA_BASELINE.ipynb first to generate it.


,check,auc,sens,spec,auc_std,n_subjects,cv_auc
0,1_baseline,0.907143,0.857143,0.850000,—,—,—
1,2_shuffled_order,0.848571,—,—,0.025813,—,—
2,3_no_delta_t,0.907143,0.857143,0.850000,—,—,—
3,4_first_2_visits,0.876623,1.000000,0.642857,—,25.000000,—
4,5_metadata_only_baseline,—,—,—,—,—,—


## Reading the table

* **Baseline vs shuffled-order.** If row 2 ≈ row 1, the LSTM is *not* exploiting temporal order — the per-visit graph embedding is doing the work, the recurrence is decorative.
* **Baseline vs Δt-removed.** If row 3 collapses, Δt is carrying real signal (which may itself be leakage; see metadata baseline).
* **Baseline vs first-2-visits.** If row 4 stays high while only seeing the *earliest* two scans, the model is doing *true early detection*. If it collapses, prior numbers were aided by later visits or by sequence-length cues.
* **Metadata-only.** Anchor for everything: subtract this from every fMRI-based row to estimate the marginal contribution of brain features.

## 6. Visit-count confound check

The early-detection AUC-vs-N table (`common/early_detection.py`) restricts to
subjects with `>= N` visits at each N, so N=1 and N=2 are scored on **different,
differently-balanced cohorts** — not the same subjects with less information. Converters
are also followed longer than stable-MCI subjects in this cohort (informative dropout),
so deeper-N cohorts are automatically converter-enriched. Both effects inflate the
apparent N=1→N=2 AUC jump independent of whatever the model actually learns from a
second scan. See `DOCS/seventh-meeting-tasks/visit-count-auc-confound.md`.

This check reuses the already-loaded checkpoint and `common/visit_confound.py` (the
same routines `SANITY_VISIT_COUNT_CONFOUND.ipynb` runs through the full adapter/runner
pipeline) via lightweight local hooks, so it works standalone against whatever
checkpoint this notebook has selected:

* **Cohort composition** — class balance at each N, to make the confound visible.
* **Fixed-cohort AUC-vs-N** — same population held constant across N; isolates
  added-visit information from the cohort/class-balance shift.
* **Spearman r(P(converter), n_scans)** — between-subject correlation of the final
  prediction with visit count.
* **Within-subject slope of P(converter)** — a flat slope alongside a strong
  between-subject correlation means the model is reading visit count as a shortcut;
  a consistent within-subject drift means it's accumulating real evidence.

The threshold used below is fixed (taken from check 1's baseline `threshold_used`) and
reused unchanged across every N, so no per-N threshold is re-derived from the data being
scored.

In [10]:
from common.crossval import Bundle
from common.early_detection import early_detection_table
from common.visit_confound import (
    summarize_visit_counts, cohort_composition_table, early_detection_fixed_cohort,
    prob_vs_visit_count, within_subject_prob_slopes,
)

TEST_BUNDLE = Bundle(
    [it['label'] for it in test_items_all],
    [it['subject_id'] for it in test_items_all],
    test_items_all,
)

def _truncate_to_n_visits(bundle, n):
    items = [
        {**it, 'graphs': it['graphs'][:n], 'delta_t': it['delta_t'][:n],
         'visit_months': it['visit_months'][:n], 'n_scans': n}
        for it in bundle.items if it['n_scans'] >= n
    ]
    return Bundle([it['label'] for it in items], [it['subject_id'] for it in items], items)

def _eval_split(state, bundle, threshold, *, device):
    return _run(bundle.items, use_time_delta=True, shuffle_order=False, threshold=threshold)

def _per_visit_probs(state, item, *, device):
    out = []
    for t in range(1, item['n_scans'] + 1):
        sub_item = {**item, 'graphs': item['graphs'][:t], 'delta_t': item['delta_t'][:t], 'n_scans': t}
        m = _run([sub_item], use_time_delta=True, shuffle_order=False, threshold=CONFOUND_THRESHOLD)
        out.append((item['visit_months'][t - 1], float(m['probs'][0])))
    return out

# Fixed threshold reused unchanged across every N (no per-N re-derivation) — taken
# from check 1's baseline evaluation, same threshold source already used elsewhere
# in this notebook.
CONFOUND_THRESHOLD = _run(test_items_all, use_time_delta=True, shuffle_order=False)['threshold_used']
print(f'Fixed threshold for check 6: {CONFOUND_THRESHOLD:.4f}')

VC_TEST = summarize_visit_counts(TEST_BUNDLE)
print('\nVisit counts by group (test split):')
display(VC_TEST)

COMP     = cohort_composition_table(TEST_BUNDLE, _truncate_to_n_visits)
ED_VAR   = early_detection_table(TEST_BUNDLE, _eval_split, _truncate_to_n_visits,
                                  model, CONFOUND_THRESHOLD, device=device)
ED_FIXED = early_detection_fixed_cohort(TEST_BUNDLE, _eval_split, _truncate_to_n_visits,
                                        model, CONFOUND_THRESHOLD, device=device)

comp_df  = pd.DataFrame(COMP)
ed_df    = pd.DataFrame(ED_VAR)
fixed_df = pd.DataFrame(ED_FIXED)
combined = (comp_df.merge(ed_df[['n_visits', 'auc']], on='n_visits', how='left')
            if not ed_df.empty else comp_df)

print('\nPer-N cohort composition + variable-cohort AUC (confounded by class-balance shift):')
display(combined)
print('Fixed-cohort AUC-vs-N (population held constant — the trustworthy number):')
display(fixed_df)

PROB_DF, SPEARMAN = prob_vs_visit_count(TEST_BUNDLE, _per_visit_probs, model, device=device)
print('\nSpearman r(P(converter), n_scans) — between-subject:')
for grp, s in SPEARMAN.items():
    print(f"  {grp:14s} r={s['r']:.3f}  p={s['p']:.3f}  n={s['n']}")

SLOPE_DF, SLOPE_STATS = within_subject_prob_slopes(TEST_BUNDLE, _per_visit_probs, model, device=device)
print('\nWithin-subject slope of P(converter) vs visit index:')
for grp, s in SLOPE_STATS.items():
    print(f"  {grp:14s} median_slope={s['median_slope']:.4f}  frac_negative={s['frac_negative']}  n={s['n']}")

rows.append({
    'check':      '6_visit_count_confound_fixed_cohort',
    'auc':        float(fixed_df['auc'].iloc[-1]) if not fixed_df.empty else float('nan'),
    'n_subjects': int(fixed_df['n_subjects'].iloc[-1]) if not fixed_df.empty else 0,
})
pd.DataFrame(rows).fillna('—')


Fixed threshold for check 6: 0.4271

Visit counts by group (test split):


,group,n,mean,median,std,min,max,mwu_pvalue
0,converter,14,2.571429,2.0,1.342460,1,5,0.677645
1,non_converter,20,2.350000,2.0,1.182103,1,4,0.677645
2,overall,34,2.441176,2.0,1.235612,1,5,0.677645



Visits    N      AUC     Sens     Spec
----------------------------------------
     1   34   0.5571    0.143    0.950


     2   25   0.8766    0.818    0.786
     3   14   1.0000    1.000    1.000


     4    8   1.0000    1.000    1.000

Visits    N      AUC     Sens     Spec
----------------------------------------
     1    8   0.5333    0.000    1.000
     2    8   0.8667    0.667    0.800
     3    8   1.0000    1.000    1.000


...

Fixed-cohort AUC-vs-N (population held constant — the trustworthy number):


,n_visits,n_subjects,auc,sensitivity,specificity
0,1,8,0.533333,0.000000,1.0
1,2,8,0.866667,0.666667,0.8
2,3,8,1.000000,1.000000,1.0
3,4,8,1.000000,1.000000,1.0



Spearman r(P(converter), n_scans) — between-subject:
  overall        r=0.043  p=0.811  n=34
  converter      r=0.906  p=0.000  n=14
  non_converter  r=-0.614  p=0.004  n=20



Within-subject slope of P(converter) vs visit index:
  overall        median_slope=0.1741  frac_negative=0.44  n=25
  converter      median_slope=0.3092  frac_negative=0.0  n=11
  non_converter  median_slope=-0.1011  frac_negative=0.7857142857142857  n=14


,check,auc,sens,spec,auc_std,n_subjects,cv_auc
0,1_baseline,0.907143,0.857143,0.85,—,—,—
1,2_shuffled_order,0.848571,—,—,0.025813,—,—
2,3_no_delta_t,0.907143,0.857143,0.85,—,—,—
3,4_first_2_visits,0.876623,1.0,0.642857,—,25.0,—
4,5_metadata_only_baseline,—,—,—,—,—,—
5,6_visit_count_confound_fixed_cohort,1.0,—,—,—,8.0,—


## Reading check 6

* **Cohort composition table.** Confirms whether N=1→N=2 drops subjects unevenly by
  class (in this cohort it drops only stable-MCI subjects, pushing converter fraction
  up) — if so, the variable-cohort AUC jump is partly/mostly a class-balance artifact,
  not new information.
* **Variable-cohort vs fixed-cohort AUC.** If the fixed-cohort curve is flat while the
  variable-cohort curve rises steeply, the rise in the variable-cohort table was cohort
  composition, not the model using the second visit. If the fixed-cohort curve also
  rises, that's the part of the effect that's real.
* **Spearman r(prob, n_scans) vs within-subject slope.** A strong *between*-subject
  correlation with a *flat* within-subject slope means the model has learned "more
  visits ⇒ more likely converter" as a shortcut (visit count as a label proxy) rather
  than reading the added scan's content. A consistent within-subject drift (e.g.
  negative slopes for stable MCI as more clean visits accumulate) is evidence
  accumulation — corroborate against check 2 (shuffled order) and check 3 (Δt removed)
  above.